In [ ]:
"""
╔══════════════════════════════════════════════════════════════════╗
║  MOLECULAR CLASSIFICATION WITH DEEP LEARNING                    ║
║  LESSON 2: Random Forest, Dynamic Random Forest                ║
║                                                                  ║
║  Two fundamentally different "classical" ML approaches.       ║
║  We compare all of them — and against Logistic Regression.     ║
╚══════════════════════════════════════════════════════════════════╝

WHAT WE LEARN TODAY:
  1. Random Forest — ensemble of decision trees (bagging)

  2. Dynamic Random Forest — ensemble of decision trees (boosting)
"""

'\n╔══════════════════════════════════════════════════════════════════╗\n║  MOLECULAR CLASSIFICATION WITH DEEP LEARNING                    ║\n║  LESSON 2: Random Forest, SVM & Gradient Boosting              ║\n║                                                                  ║\n║  Three fundamentally different "classical" ML approaches.       ║\n║  We compare all of them — and against Logistic Regression.     ║\n╚══════════════════════════════════════════════════════════════════╝\n\nWHAT WE LEARN TODAY:\n  1. Random Forest — ensemble of decision trees (bagging)\n  2. Support Vector Machine — margin maximisation + kernel trick\n  3. Gradient Boosting — sequential error correction\n  4. Full benchmark comparison: all 4 classifiers side-by-side\n  5. Learning curves — how much data do we need?\n'

In [13]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import re, math
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, cross_val_score, learning_curve
from sklearn.metrics import (classification_report, roc_auc_score,
                              roc_curve, confusion_matrix)
from sklearn.preprocessing import StandardScaler
from sklearn.inspection import permutation_importance
import pickle, warnings
warnings.filterwarnings('ignore')

In [5]:
plt.style.use('dark_background')
COLORS = {'LR': '#38bdf8', 'RF': '#34d399', 'SVM': '#a78bfa',
          'GB': '#fb923c', 'pos': '#34d399', 'neg': '#f87171',
          'gold': '#fbbf24'}


df = pd.read_csv('mol_dataset.csv')



In [8]:
df.head()
print(f"Dataset: {len(df)} molecules")
print(f"BBB Positive (crosses): {df.bbb.sum()}  ({df.bbb.mean()*100:.0f}%)")
print(f"BBB Negative (blocked): {(1-df.bbb).sum()}  ({(1-df.bbb.mean())*100:.0f}%)")
print(f"\nSample molecules:")
print(df[['name','smiles','bbb']].head(6).to_string(index=False))


Dataset: 49 molecules
BBB Positive (crosses): 25  (51%)
BBB Negative (blocked): 24  (49%)

Sample molecules:
     name                                  smiles  bbb
 Caffeine              Cn1cnc2c1c(=O)n(C)c(=O)n2C    1
  Aspirin                   CC(=O)Oc1ccccc1C(=O)O    1
Ibuprofen              CC(C)Cc1ccc(cc1)C(C)C(=O)O    1
 Diazepam     CN1C(=O)CN=C(c2ccccc2)c3cc(Cl)ccc13    1
 Morphine       OC1CC2CC1N(C)CCC3=CC(=C(C=C23)O)O    1
      THC CCCCCc1cc2c(cc1OC)OC(C)(C)c1ccc(C)cc1-2    1


In [9]:
#building our own feature vector not using rdkit :


# ═══════════════════════════════════════════════════════════════════
# STEP 2 — SMILES FEATURISATION (Building Our Own "Fingerprint")
# ═══════════════════════════════════════════════════════════════════
print("\n" + "=" * 65)
print("  STEP 2: SMILES → FEATURE VECTOR")
print("=" * 65)
print("""
A SMILES string encodes rich chemical information as text.
We extract features from this text — just like NLP feature engineering.

FEATURE CATEGORIES:
  A) Atom counts       — how many C, N, O, S, F, Cl, Br atoms
  B) Bond/structure    — rings, double bonds, aromatic bonds
  C) Physicochemical   — molecular weight (approx), logP proxy
  D) Text n-grams      — character bigrams (Jardim et al. approach)
  E) Rule-of-5 flags   — Lipinski drug-likeness rules
""")


  STEP 2: SMILES → FEATURE VECTOR

A SMILES string encodes rich chemical information as text.
We extract features from this text — just like NLP feature engineering.

FEATURE CATEGORIES:
  A) Atom counts       — how many C, N, O, S, F, Cl, Br atoms
  B) Bond/structure    — rings, double bonds, aromatic bonds
  C) Physicochemical   — molecular weight (approx), logP proxy
  D) Text n-grams      — character bigrams (Jardim et al. approach)
  E) Rule-of-5 flags   — Lipinski drug-likeness rules



In [10]:
# Atomic weights for MW estimation
ATOM_WEIGHTS = {'C': 12.0, 'N': 14.0, 'O': 16.0, 'S': 32.0,
                'F': 19.0, 'Cl': 35.5, 'Br': 80.0, 'I': 127.0,
                'P': 31.0, 'H': 1.0}
#these weights are used for a quick estimation of molecular weight based on atom counts.
#how did we chose these weights? They are rounded atomic weights of the elements commonly found in drug-like molecules. Sources ? IUPAC and standard atomic weights from chemistry references.
#  We include H (hydrogen) for better MW estimation, even though it's often implicit in SMILES. 
# This allows us to estimate the molecular weight without needing a full chemistry toolkit, which is useful for our custom featurisation approach.

In [11]:

# Hydrophobicity contribution (Crippen logP fragments — simplified)
LOGP_CONTRIB = {'C': 0.53, 'c': 0.13, 'N': -1.03, 'n': -0.57,
                'O': -0.67, 'o': -0.32, 'S': 0.03, 'F': 0.14,
                'Cl': 0.60, 'Br': 0.88, 'I': 1.35, 'P': 0.22}

# These contributions are based on the Crippen logP fragment values, which assign a hydrophobicity contribution to each atom type.
# We use these values to estimate the overall logP of the molecule, which is a key factor in BBB permeability.
# The values are simplified and may not capture all nuances, but they provide a useful proxy for our featurisation without needing a full chemistry toolkit.


In [14]:
def smiles_to_features(smiles):
    """
    Convert a SMILES string to a numerical feature vector.
    
    This is the core of feature-engineered molecular classification.
    Every feature has a chemical meaning.
    """
    s = smiles

    # ── A: ATOM COUNTS ─────────────────────────────────────────────
    n_C  = len(re.findall(r'[Cc]', s)) # count both uppercase and lowercase C for total carbon count in the string s.
    n_N  = len(re.findall(r'[Nn]', s))
    n_O  = len(re.findall(r'[Oo]', s))
    n_S  = len(re.findall(r'[Ss]', s))
    n_F  = s.count('F')
    n_Cl = s.count('Cl')
    n_Br = s.count('Br')
    n_I  = s.count('I')
    n_P  = s.count('P')

    # ── B: BOND & TOPOLOGY FEATURES ────────────────────────────────
    n_double   = s.count('=')          # double bonds
    n_triple   = s.count('#')          # triple bonds
    n_aromatic = len(re.findall(r'[a-z]', s))  # lowercase = aromatic
    n_rings    = len(re.findall(r'\d', s)) // 2  # ring closure digits
    n_branches = s.count('(')          # branch points
    n_charged  = s.count('+') + s.count('-')    # charges

    # ── C: MOLECULAR WEIGHT (approximate) ─────────────────────────
    mw = 0.0
    for atom, wt in ATOM_WEIGHTS.items():
        if len(atom) == 2:
            mw += s.count(atom) * wt
            #why? We check for two-character atoms (like Cl, Br) first to avoid double-counting the 'C' and 'l' in 'Cl' or 'B' and 'r' in 'Br'.
            #  If we counted single characters first, we would count the 'C' in 'Cl' as a carbon atom, which would inflate our carbon count
            #  and thus our molecular weight estimation. By counting two-character atoms first, we ensure that we correctly identify and count these specific elements without confusion.
        else:
            # single-letter — avoid double-counting Cl, Br
            count = len(re.findall(f'(?<![A-Z]){atom}(?!l|r)', s))
            mw += count * wt
    # Add implicit hydrogens (rough estimate)
    n_heavy = n_C + n_N + n_O + n_S + n_F + n_Cl + n_Br + n_I + n_P #why? We calculate the total number of heavy atoms (non-hydrogen) in the molecule.
    #This is done by summing the counts of all the atoms we identified in the previous steps.
    mw += n_heavy * 1.0 * 1.5  # H approximation

    # ── D: logP PROXY (hydrophilicity / lipophilicity) ─────────────
    logp = sum(LOGP_CONTRIB.get(c, 0) for c in s)

    # ── E: POLAR SURFACE AREA proxy (H-bond donors/acceptors) ──────
    hbd = len(re.findall(r'[OH]|N(?!.*\()', s))   # H-bond donors
    hba = len(re.findall(r'[NO]', s))              # H-bond acceptors

    # ── F: LIPINSKI RULE-OF-5 FLAGS ────────────────────────────────
    # Oral bioavailability proxy — MW<500, logP<5, HBD≤5, HBA≤10
    ro5_mw   = 1 if mw < 500 else 0
    ro5_logp = 1 if logp < 5  else 0
    ro5_hbd  = 1 if hbd <= 5  else 0
    ro5_hba  = 1 if hba <= 10 else 0
    ro5_pass  = ro5_mw + ro5_logp + ro5_hbd + ro5_hba  # 0-4 rules passed

    # ── G: TEXT N-GRAMS (Jardim et al. approach) ───────────────────
    # Count character bigrams in the SMILES string
    # This captures local chemical context without explicit parsing
    BIGRAMS = ['CC', 'CN', 'CO', 'CS', 'C=', 'C(', 'C1', 'c1', 'c2',
               'cc', 'NC', 'OC', 'O=', 'N=', 'Cl', 'Br', '=O', '=N',
               'C#', 'c(', 'n1', 'N1', 'OC', 'SC', 'FC', 'C)', 'O)']
    bigram_counts = [s.count(bg) for bg in BIGRAMS]

    # ── H: STRUCTURAL COMPLEXITY ────────────────────────────────────
    smiles_len   = len(s)
    unique_atoms = len(set(re.findall(r'[A-Za-z]', s)))
    frac_arom    = n_aromatic / max(smiles_len, 1)  # aromaticity fraction

    # Assemble feature vector
    features = [
        # Atom counts (9)
        n_C, n_N, n_O, n_S, n_F, n_Cl, n_Br, n_I, n_P,
        # Bonds/topology (6)
        n_double, n_triple, n_aromatic, n_rings, n_branches, n_charged,
        # Physicochemical (5)
        mw, logp, hbd, hba, ro5_pass,
        # Lipinski flags (4)
        ro5_mw, ro5_logp, ro5_hbd, ro5_hba,
        # Complexity (3)
        smiles_len, unique_atoms, frac_arom,
        # Bigrams (28)
        *bigram_counts
    ]
    return np.array(features, dtype=float)




In [15]:
FEATURE_NAMES = (
    ['n_C','n_N','n_O','n_S','n_F','n_Cl','n_Br','n_I','n_P'] +
    ['n_double','n_triple','n_aromatic','n_rings','n_branches','n_charged'] +
    ['mol_weight','logP_proxy','HBD','HBA','ro5_passed'] +
    ['ro5_mw','ro5_logp','ro5_hbd','ro5_hba'] +
    ['smiles_len','unique_atoms','frac_aromatic'] +
    [f'bigram_{i}' for i in range(28)]
)
# Apply to all molecules
X = np.array([smiles_to_features(s) for s in df['smiles']])
y = df['bbb'].values

print(f"Feature matrix shape: {X.shape}")
print(f"  → {X.shape[0]} molecules × {X.shape[1]} features")
print(f"\nExample: Aspirin features (first 20)")
aspirin_idx = df[df.name == 'Aspirin'].index[0]
for name, val in zip(FEATURE_NAMES[:20], X[aspirin_idx, :20]):
    print(f"  {name:<20} = {val:.2f}")

Feature matrix shape: (49, 54)
  → 49 molecules × 54 features

Example: Aspirin features (first 20)
  n_C                  = 9.00
  n_N                  = 0.00
  n_O                  = 4.00
  n_S                  = 0.00
  n_F                  = 0.00
  n_Cl                 = 0.00
  n_Br                 = 0.00
  n_I                  = 0.00
  n_P                  = 0.00
  n_double             = 2.00
  n_triple             = 0.00
  n_aromatic           = 6.00
  n_rings              = 1.00
  n_branches           = 2.00
  n_charged            = 0.00
  mol_weight           = 107.50
  logP_proxy           = -0.31
  HBD                  = 4.00
  HBA                  = 4.00
  ro5_passed           = 4.00


In [6]:
print("\n" + "=" * 65)
print("  ALGORITHM 2: RANDOM FOREST")
print("=" * 65)
print("""
IDEA: Train N independent decision trees, each on a random subset 
of data AND features. Predict by majority vote.

WHY RANDOM?
  → Each tree sees different data  (bootstrap sampling = "bagging")
  → Each split considers random subset of features (reduces correlation)
  → Many weak learners → one strong learner

FOR MOLECULES:
  → Each tree learns different substructure rules
  → Tree 1: "if n_aromatic > 3 AND logP > 2 → BBB+"
  → Tree 2: "if mol_weight < 300 AND n_O < 3 → BBB+"
  → Final prediction: majority vote of all trees

KEY HYPERPARAMETERS:
  n_estimators   = number of trees (more = better, but slower)
  max_depth      = how deep each tree grows (controls overfitting)
  max_features   = features considered per split (sqrt(n) default)
""")


  ALGORITHM 2: RANDOM FOREST

IDEA: Train N independent decision trees, each on a random subset 
of data AND features. Predict by majority vote.

WHY RANDOM?
  → Each tree sees different data  (bootstrap sampling = "bagging")
  → Each split considers random subset of features (reduces correlation)
  → Many weak learners → one strong learner

FOR MOLECULES:
  → Each tree learns different substructure rules
  → Tree 1: "if n_aromatic > 3 AND logP > 2 → BBB+"
  → Tree 2: "if mol_weight < 300 AND n_O < 3 → BBB+"
  → Final prediction: majority vote of all trees

KEY HYPERPARAMETERS:
  n_estimators   = number of trees (more = better, but slower)
  max_depth      = how deep each tree grows (controls overfitting)
  max_features   = features considered per split (sqrt(n) default)



In [16]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [17]:
rf = RandomForestClassifier(
    n_estimators=200,    # 200 trees
    max_depth=None,      # grow until pure — RF handles overfitting via ensemble
    max_features='sqrt', # sqrt(54) ≈ 7 features per split
    min_samples_leaf=1,
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train)  # RF doesn't need scaled features

y_pred_rf = rf.predict(X_test)
y_prob_rf  = rf.predict_proba(X_test)[:, 1]
auc_rf     = roc_auc_score(y_test, y_prob_rf)
cv_rf      = cross_val_score(rf, X, y, cv=5, scoring='roc_auc').mean()


In [18]:
print(f"Random Forest Results:")
print(f"  Test Accuracy : {(y_pred_rf == y_test).mean():.3f}")
print(f"  Test ROC-AUC  : {auc_rf:.3f}")
print(f"  5-fold CV AUC : {cv_rf:.3f}")
print(f"\nClassification Report:")
print(classification_report(y_test, y_pred_rf, target_names=['BBB-', 'BBB+']))

Random Forest Results:
  Test Accuracy : 0.800
  Test ROC-AUC  : 0.880
  5-fold CV AUC : 0.890

Classification Report:
              precision    recall  f1-score   support

        BBB-       0.80      0.80      0.80         5
        BBB+       0.80      0.80      0.80         5

    accuracy                           0.80        10
   macro avg       0.80      0.80      0.80        10
weighted avg       0.80      0.80      0.80        10



In [19]:
# Feature importance (built-in Gini importance)
feat_imp = rf.feature_importances_
FEAT_NAMES = (
    ['n_C','n_N','n_O','n_S','n_F','n_Cl','n_Br','n_I','n_P'] +
    ['n_double','n_triple','n_aromatic','n_rings','n_branches','n_charged'] +
    ['mol_weight','logP_proxy','HBD','HBA','ro5_passed'] +
    ['ro5_mw','ro5_logp','ro5_hbd','ro5_hba'] +
    ['smiles_len','unique_atoms','frac_aromatic'] +
    [f'bigram_{i}' for i in range(28)]
)
top_rf = np.argsort(feat_imp)[::-1][:10]
print(f"\nTop 10 features (Gini importance):")
for i in top_rf:
    print(f"  {FEAT_NAMES[i]:<22} {feat_imp[i]:.4f}  {'█'*int(feat_imp[i]*200)}")


Top 10 features (Gini importance):
  bigram_26              0.1369  ███████████████████████████
  HBA                    0.1082  █████████████████████
  bigram_10              0.0686  █████████████
  n_O                    0.0611  ████████████
  logP_proxy             0.0573  ███████████
  frac_aromatic          0.0540  ██████████
  HBD                    0.0520  ██████████
  mol_weight             0.0375  ███████
  n_aromatic             0.0375  ███████
  bigram_8               0.0356  ███████
